Die benötigten Bibliotheken werden importiert

In [1]:
from random import seed, randint
seed(23)

import simpy

Die Klasse ElektronischesFahrzeug wird definiert. 
Der Prozess batterie_steuerung wird in der init methode aufgerufen und und direkt nach der while Schleife passivated. 
Nachdem dieser in der Mehode drive() wieder reaktiviert wird, läuft dieser Prozess weiter, bis er in die nächste While-Schleifen-Iteration gelangt. 

In [2]:
class ElektronischesFahrzeug:
    def __init__(self, env: simpy.Environment):
        self.env = env
        self.drive_proc = env.process(self.drive(env))
        self.batterie_steuerungs_prozess = env.process(self.batterie_steuerung(env))
        self.batterie_steuerung_reactivate = env.event()

    def drive(self, env: simpy.Environment):
        """Das Fahrzeug fährt 20-40 min und startet danach das Parken für 1-6 h. 
        Während des parkens wird die Methode batterie_steuerung aufgerufen."""
        while True:
            # Fahre für 20-40 min
            yield env.timeout(randint(20,40))

            # Parke für 1-6 h
            print("Starte das parken um: ", env.now)
            self.batterie_steuerung_reactivate.succeed() # reactivate
            self.batterie_steuerung_reactivate = env.event()
            yield env.timeout(randint(60,360))
            print("Parken endet um:", env.now)


    def batterie_steuerung(self, env: simpy.Environment):
        while True:
            print("Batteriesteuerung wurde passivated um:", env.now)
            yield self.batterie_steuerung_reactivate # passivate
            print("Batteriesteuerung wurde reactivated um:", env.now)

            # Eigentliche Logik der Methode

            yield env.timeout(randint(30, 90))


env = simpy.Environment()
ef = ElektronischesFahrzeug(env)

env.run(until=150)


Batteriesteuerung wurde passivated um: 0
Starte das parken um:  29
Batteriesteuerung wurde reactivated um: 29
Batteriesteuerung wurde passivated um: 60
Parken endet um: 131
